# 02 - Data Preprocessing

**Coursework Part 01, Task 2.I**

**Goal:** turn the raw Kaggle file into a clean, consistent dataset and a stratified train/test split that the modelling notebook can use directly. Each step is justified using what was found in the EDA notebook.

**What happens here and what does not**

| Done in this notebook (data-level fixes) | Done in the modelling notebook |
|---|---|
| Fix data types, handle missing values, remove the ID, standardise categories, check duplicates and outliers, encode the target, train/test split | One-hot encoding, and scaling |

Encoding, and scaling *learn* something from the data (category lists, means, standard deviations, neighbours). If they were fitted on the full dataset before splitting, information from the test set would leak into training. Below section show how they work, fitted on the training set only, but the saved files stay unencoded so the pipeline can refit them inside each cross-validation fold.

## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

sns.set_theme(style = "whitegrid", context = "notebook")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
TEST_SIZE = 0.2

RAW_PATH = Path("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
CLEAN_PATH = Path("../data/processed/telco_churn_clean.csv")
TRAIN_PATH = Path("../data/processed/train.csv")
TEST_PATH = Path("../data/processed/test.csv")

## Load The Raw Data

In [2]:
df_raw = pd.read_csv(RAW_PATH)
df = df_raw.copy()   # keep the original untouched for before/after comparisons

print(f"Raw shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Raw shape: 7,043 rows x 21 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Fix Data Types

`TotalCharges` should be numeric but is stored as text (`object`/`str`). Converting it with `errors="coerce"` turns any value that is not a valid number into `NaN`, which exposes the hidden problem values.

In [3]:
print("TotalCharges dtype before:", df["TotalCharges"].dtype)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("TotalCharges dtype after :", df["TotalCharges"].dtype)
print("Values that could not be converted:", df["TotalCharges"].isna().sum())

TotalCharges dtype before: str
TotalCharges dtype after : float64
Values that could not be converted: 11


## Handle Missing Values

In [4]:
missing = df.isna().sum()
missing[missing > 0]

TotalCharges    11
dtype: int64

All 11 blank values belong to the only 11 customers with tenure = 0. They have just joined and have not received a bill yet, so their true total charge is 0. This is not a guess.

In [5]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)
print("Missing values remaining:", df.isna().sum().sum())

Missing values remaining: 0


## Remove The Identifier Column

In [6]:
print("Unique customerIDs:", df["customerID"].nunique(), "of", len(df))
df = df.drop(columns="customerID")

Unique customerIDs: 7043 of 7043


## Standardize Categorical Values

In [7]:
cat_cols = df.select_dtypes(exclude="number").columns.tolist()

# Check for stray whitespace or inconsistent capitalisation
for col in cat_cols:
    values = df[col].unique()
    stripped = {v.strip() for v in values}
    lowered = {v.strip().lower() for v in values}
    if len(stripped) != len(values) or len(lowered) != len(values):
        print(f"Inconsistent values in {col}: {values}")
print("Whitespace/case check complete.")

Whitespace/case check complete.


In [8]:
# SeniorCitizen is the only binary feature stored as 0/1; every other one uses Yes/No.
# Converting it means all categorical features are treated the same way by the encoder.
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
df["SeniorCitizen"].value_counts()

SeniorCitizen
No     5901
Yes    1142
Name: count, dtype: int64